# 00 — Build evaluation dataset

Mục tiêu: xây 50 câu Q–A vàng trên 5 nhóm (10 câu / nhóm). Dataset này là nền tảng cho MỌI RQ khác.

**Nhóm** (xem `research/data/categories.json`):
- `simple_penalty` — mức phạt đơn giản
- `multi_intent` — câu đa ý
- `cross_reference` — tham chiếu chéo điểm/khoản
- `procedure` — thủ tục / đăng kiểm
- `out_of_scope` — control

**Output:** `research/data/eval_qa.jsonl`

Quy trình curate:
1. Random sample Điều từ `all_chunks.jsonl` cho mỗi nhóm → suggest câu hỏi.
2. Tự viết `gold_answer` (1–3 câu) dựa trên chunk gốc.
3. Ghi `gold_citations` ở mức khoản/điểm.
4. Đồng nghiệp review (nếu có) để giảm bias.

In [ ]:
# --- Setup ---
import sys
from pathlib import Path

HERE = Path.cwd()
# Walk up to find traffic_rag/ (works when run from anywhere under it)
for p in [HERE] + list(HERE.parents):
    if (p / "source").is_dir() and (p / "research").is_dir():
        TRAFFIC_RAG = p
        break
else:
    raise RuntimeError("Could not locate traffic_rag/ root")

sys.path.insert(0, str(TRAFFIC_RAG))
sys.path.insert(0, str(TRAFFIC_RAG.parent))  # so `research.utils` works

from research.utils.langsmith_setup import enable_tracing
enable_tracing(project="traffic-rag-research", run_name="rq0-build-eval")

EVAL_PATH = TRAFFIC_RAG / "research" / "data" / "eval_qa.jsonl"
RESULTS_DIR = TRAFFIC_RAG / "research" / "results" / "metrics"
FIGURES_DIR = TRAFFIC_RAG / "research" / "results" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load corpus & sample chunks cho mỗi nhóm

In [ ]:
# TODO: load all_chunks.jsonl, đếm phân bố Điều/Khoản, random-sample
import json

CHUNKS_PATH = TRAFFIC_RAG / "Data" / "all_chunks.jsonl"
chunks = [json.loads(l) for l in CHUNKS_PATH.open(encoding="utf-8") if l.strip()]
print("total chunks:", len(chunks))
# TODO: group by doc_id + dieu, pick representative samples per category


## 2. Write / edit eval_qa.jsonl

In [ ]:
# TODO: append curated rows to research/data/eval_qa.jsonl
# Schema: {id, category, question, gold_answer, gold_citations, expected_category, notes}
# Keep the 5 seed rows already in the file as examples.


## 3. Sanity check — phân bố, độ dài, trùng lặp

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in EVAL_PATH.open(encoding="utf-8") if l.strip()]
df = pd.DataFrame(rows)
print(df["category"].value_counts())
print("avg question len (tokens):", df["question"].str.split().str.len().mean())
assert df["id"].is_unique, "Duplicate ids!"


## Kết luận

_TODO: ghi chú về quy trình curate, tỉ lệ bias, thời gian bỏ ra._